In [24]:
import os
import pickle
import plotly.io as pio
pio.renderers.default = "notebook_connected"
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [25]:
# from google.colab import drive
# drive.mount('/content/drive')

In [26]:
# Charger les données

data_path = "../data/processed"
file_name = 'dataset_final_phrases.pkl'
with open(os.path.join(data_path, file_name), 'rb') as f:
    df = pickle.load(f)

In [27]:
# Charger un seul embedding

embeddings_path = "../data/embeddings"
# file_name = 'emb_sbert_multi.pkl'

# with open(os.path.join(embeddings_path, file_name), 'rb') as f:
#     embedding = pickle.load(f)

In [28]:
# Charger tous les embeddings

# embeddings_path = "../data/embeddings"
# embeddings = {}

# for file_name in os.listdir(embeddings_path):
#     with open(os.path.join(embeddings_path, file_name), 'rb') as f:
#         emb_name = file_name.split('.')[0]
#         embeddings[emb_name] = pickle.load(f)

### BERTopic

In [47]:
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
import nltk
from nltk.corpus import stopwords
from hdbscan import HDBSCAN
import umap
import numpy as np
from sklearn.cluster import KMeans
import pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import normalize
import warnings

In [30]:
sentences = df["sentence"].tolist()

file_name_mutli = 'emb_sbert_multi_phrases.pkl'
with open(os.path.join(embeddings_path, file_name_mutli), 'rb') as f:
    embedding_multi = pickle.load(f)

file_name_fr = 'emb_sbert_fr_phrases.pkl'
with open(os.path.join(embeddings_path, file_name_fr), 'rb') as f:
    embedding_fr = pickle.load(f)

file_name_perf = 'emb_sbert_multi2_phrases.pkl'
with open(os.path.join(embeddings_path, file_name_fr), 'rb') as f:
    embedding_multi2 = pickle.load(f)

In [31]:
print(len(sentences))
print(embedding_multi.shape)
print(embedding_fr.shape)
print(embedding_multi2.shape)

37628
(37628, 384)
(37628, 768)
(37628, 768)


In [46]:
warnings.filterwarnings("ignore", category=UserWarning)

product_words = ['montre', 'montres', 'boucle', 'boucles', 'oreilles', 'oreille', 'paire', 'paires', 'lunettes', 'bracelet', 'bracelets', 'collier', 'iphone', 'téléphone', 'pendentif', 'robot', 'robots', 'aspirateur', 'baskets', 'basket', 'chaussure', 'chaussures', 'sandales', 'plantes', 'plante', 'arbres', 'arbre', 'bulbes', 'willemse', 'sommiers', 'jardin', 'bague', 'lampe', 'lampes', 'abat-jour', 'abat jour', 'lampadaire', 'parfum', 'shampoing', 'shampooing', 'shampooings', 'cheveux', 'masque', 'masques', 'crème', 'élastiques', 'manteau', 'bougie', 'cadre', 'écouteurs', 'vélo', 'robe', 'vêtements', 'bijoux', 'sac', 'portable', 'clio', 'luminaire', 'oreillette', 'induction', 'écouteur', 'couette', 'samsung', 'téléphones', 'smartcase', 'abat', 'apple', 'watch', 'shirt', 'tee', 'chemise', 'shirts', 'hortensias', 'orchidée', 'sacs', 'plant', 'reconditionné']

models = {}

for embedding, name in zip([embedding_fr, embedding_multi, embedding_multi2], ["français", "multilingue", "multilingue_2"]):
    print(name)

    vectorizer_model = CountVectorizer(
        stop_words=stopwords.words("french") + product_words,
        ngram_range=(1, 3),
        #min_df=2,
        max_df=0.95
    )
    # ctfidf_model = ClassTfidfTransformer()

    hdbscan_model = HDBSCAN(
    min_cluster_size=10,
    min_samples=3
    )
    
    topic_model = BERTopic(
        language="french",
        vectorizer_model=vectorizer_model,
        hdbscan_model=hdbscan_model,
        verbose=False,
        nr_topics="auto"
        # ctfidf_model=ctfidf_model
    )
    
    topics, probs = topic_model.fit_transform(sentences, embedding)

    # topic_model.reduce_topics(sentences, nr_topics=30)
    
    topics = np.array(topics)
    
    # Identifier les gros clusters
    topic_sizes = pd.Series(topics).value_counts()
    large_topics = topic_sizes[topic_sizes > 2000].index  # seuil à ajuster selon dataset
    
    for t in large_topics:
        print("Topic :", t)
        # Récupérer les indices des documents dans le gros cluster
        idx = np.where(topics == t)[0]
        embeddings_subset = embedding[idx]
        
        # Diviser le gros cluster avec KMeans
        n_subclusters = int(len(idx) / 200)  # ~200 docs par sous-cluster
        print("Nombre de clusters créés par k-means :", n_subclusters)
        
        emb_norm = normalize(embeddings_subset)
        kmeans = MiniBatchKMeans(
            n_clusters=n_subclusters,
            batch_size=512,
            max_iter=200,
            n_init="auto"
        )
        sub_labels = kmeans.fit_predict(emb_norm)
        
        # Réassigner les labels dans `topics`
        max_topic_id = topics.max() + 1
        for i, doc_idx in enumerate(idx):
            topics[doc_idx] = max_topic_id + sub_labels[i]

        topic_model.update_topics(
            docs=sentences,
            topics=topics,
            vectorizer_model=topic_model.vectorizer_model,
            top_n_words=15
        )

    models[name] = topic_model

français
Topic : -1
Nombre de clusters créés par k-means : 106


2025-11-20 19:01:16,325 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Topic : 0
Nombre de clusters créés par k-means : 38


2025-11-20 19:01:20,740 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


multilingue
Topic : -1
Nombre de clusters créés par k-means : 77


2025-11-20 19:01:41,554 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


multilingue_2
Topic : -1
Nombre de clusters créés par k-means : 112


2025-11-20 19:02:01,395 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Topic : 0
Nombre de clusters créés par k-means : 73


2025-11-20 19:02:05,634 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


### Evaluation

#### Diversité

In [48]:
from itertools import chain

def topic_diversity(topic_model, top_n=10):
    """
    Calcule la diversité des topics d'un modèle BERTopic.
    """
    topics = topic_model.get_topics()

    # On extrait les mots uniquement (sans les scores)
    topic_words = []
    for topic_id, word_scores in topics.items():
        # BERTopic place les topics -1 et autres meta-topics, donc on ignore topic -1
        if topic_id == -1:
            continue
        top_words = [w for (w, score) in word_scores[:top_n]]
        topic_words.append(top_words)

    # Liste aplatie
    all_words = list(chain.from_iterable(topic_words))
    unique_words = set(all_words)

    return len(unique_words) / len(all_words)


# Calcul du score pour chaque modèle
diversity_scores = {}

for name, model in models.items():
    score = topic_diversity(model, top_n=10)
    diversity_scores[name] = score

diversity_scores

{'français': 0.6946428571428571,
 'multilingue': 0.7632692307692308,
 'multilingue_2': 0.5084577114427861}

#### Score de cohérence

In [49]:
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
from collections import defaultdict
import numpy as np
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

for name, model in models.items():
    topics = model.topics_
    docs_by_topic = defaultdict(list)
    for doc, t in zip(sentences, topics):
        if t != -1:
            docs_by_topic[t].append(doc)
    
    # Générer top words manuellement
    topic_words = []
    for t, docs in docs_by_topic.items():
        vec = TfidfVectorizer(stop_words=stopwords.words("french") + product_words, ngram_range=(1,3))
        X = vec.fit_transform(docs)
        feature_names = np.array(vec.get_feature_names_out())
        # tfidf_sum = X.toarray().sum(axis=0)
        tfidf_sum = np.asarray(X.sum(axis=0)).ravel()
        top_words = feature_names[np.argsort(tfidf_sum)[::-1]][:10].tolist()
        topic_words.append(top_words)
    
    # Tokenisation des documents
    tokenized_docs = [doc.lower().split() for doc in sentences]
    dictionary = Dictionary(tokenized_docs)
    
    # Calcul du score de cohérence
    cm = CoherenceModel(
        topics=topic_words,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )
    score = cm.get_coherence()
    print(name, score)

français 0.5219372121080242
multilingue 0.5077983464117374
multilingue_2 0.5430946881683159


#### Embedding-based coherence score

In [50]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def embedding_coherence(topic_model, embedder, top_n=10):
    """
    Calcule la cohérence basée sur les embeddings pour un modèle BERTopic.
    embedder : modèle sentence-transformers pour transformer les mots en vecteurs.
    """
    topics = topic_model.get_topics()
    scores = []

    for topic_id, word_scores in topics.items():
        if topic_id == -1:
            continue
        top_words = [w for w, _ in word_scores[:top_n]]
        word_embeddings = embedder.encode(top_words)
        sim_matrix = cosine_similarity(word_embeddings)
        
        # On enlève la diagonale (sim = 1)
        n = len(top_words)
        if n > 1:
            sims = (sim_matrix.sum() - n) / (n*(n-1))  # moyenne des cosinus
            scores.append(sims)

    return np.mean(scores)

# Exemple avec un modèle SentenceTransformer
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer('all-MiniLM-L6-v2')

for name, model in models.items():
    score = embedding_coherence(model, embedder, top_n=10)
    print(f"{name} - Embedding-based coherence: {score:.4f}")

français - Embedding-based coherence: 0.4020
multilingue - Embedding-based coherence: 0.4044
multilingue_2 - Embedding-based coherence: 0.3553


### Stockage

In [60]:
# for name, model in models.items():
#     with open(f"../models/topic_modeling/bertopic_{name}_phrases.pkl", "wb") as f:
#         pickle.dump(model, f)

In [61]:
# df["topics"] = models["multilingue_2"].topics_
# df.to_csv("./artifacts/bertopic/reviews_phrases_with_topics.csv", index=False, encoding="utf8")

In [34]:
import pandas as pd

# Charger le fichier CSV
df_avis = pd.read_csv("./artifacts/bertopic/reviews_with_topics.csv")

# Charger le fichier excel avec les nouveaux groupes
df_clusters = pd.read_excel("./artifacts/bertopic/topics_top_words.xlsx")

# Fusionner les fichiers sur la colonne du cluster
df_merged = df_avis.merge(
    df_clusters[["Topic", "Category"]],
    left_on='topics', # nom dans le CSV
    right_on='Topic', # nom dans l'Excel
    how='left'
)
df_merged = df_merged.drop(columns=['Topic'])

df_merged = df_merged[df_merged['Category'] != 'neutre']

map_clusters = {
    "qualité produit": 0,
    "livraison": 1,
    "service client": 2
}

df_merged['Label'] = df_merged['Category'].map(map_clusters)

# Exporter la nouvelle version
df_merged.to_csv("../resultats/bertopic/data/dataset_labellise_phrases.csv", index=False, encoding="utf8")

### Visualisation

In [10]:
# CHARGER LES MODELES ENREGISTRES
# models = {}
# for name in ["français", "multilingue", "multilingue_2"]:
#     with open(os.path.join("../models/topic_modeling/", f"bertopic_{name}.pkl"), 'rb') as f:
#         models[name] = pickle.load(f)

In [37]:
all_topics_words = {}
topic_model = models["multilingue_2"]

for topic_id in topic_model.get_topics().keys():
    if topic_id == -1:
        continue  # ignorer les outliers
    all_topics_words[topic_id] = [word for word, _ in topic_model.get_topic(topic_id)]

all_topics_words

{0: ['commande',
  'plus',
  'livraison',
  'très',
  'site',
  'service',
  'colis',
  'client',
  'vente',
  'service client'],
 1: ['attendais',
  'conforme',
  'très',
  'qualité',
  'rendez',
  'produit',
  'satisfaite',
  'qualité rendez',
  'article',
  'correspond'],
 2: ['conforme',
  'description',
  'conformes',
  'conforme commande',
  'conforme description',
  'photo',
  'produits conformes',
  'conforme photo',
  'conformes commande',
  'conformes description'],
 3: ['livraison temps',
  'temps',
  'temps livraison',
  'délais',
  'livraison délais',
  'livraison',
  'livraison temps livraison',
  'temps livraison temps',
  'délais livraison',
  'temps livraison délais'],
 4: ['attentes',
  'conforme attentes',
  'conforme',
  'attentes produit',
  'attentes commande',
  'attentes très',
  'produit conforme attentes',
  'commande conforme attentes',
  'produit conforme',
  'correspondant'],
 5: ['rien dire',
  'dire',
  'rien',
  'rien dire rien',
  'dire rien',
  'rien r

In [51]:
# topics trouvés
models["français"].get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,1,194,1_fuir_voleurs_mefiez site_deux pieds,"[fuir, voleurs, mefiez site, deux pieds, décon...",[166880895je déconseille fortement se site a f...
1,2,184,2_satisfaite_satisfaite commande_très satisfai...,"[satisfaite, satisfaite commande, très satisfa...","[très satisfaite de mon achat, je suis très sa..."
2,3,183,3_commandé_attends_payé_passé,"[commandé, attends, payé, passé, février, bobo...",[après l'achat d'un canapé de la marque boboch...
3,4,161,4_rapide_livraison rapide_top_top livraison,"[rapide, livraison rapide, top, top livraison,...","[livraison rapide et bon produit, la livraison..."
4,5,138,5_satisfaite_avant date_très satisfaite_date,"[satisfaite, avant date, très satisfaite, date...",[très satisfait date de livraison respecté mêm...
...,...,...,...,...,...
443,444,139,444_2020_12_11_mai,"[2020, 12, 11, mai, juin, 18, livraison prévue...",NaN
444,445,46,445_frais port_port_frais_retours,"[frais port, port, frais, retours, frais retou...",NaN
445,446,182,446_acheter sans hésiter_après litige_diffuser...,"[acheter sans hésiter, après litige, diffuser,...",NaN
446,447,282,447_dit_service client_client_service,"[dit, service client, client, service, colis, ...",NaN


In [52]:
# topics trouvés
models["multilingue"].get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,774,0_showroom_showroomprive_showroomprivé_showroo...,"[showroom, showroomprive, showroomprivé, showr...","[je suis très déçu par showroom privé ., cette..."
1,1,762,1_euros_50_bon achat_frais,"[euros, 50, bon achat, frais, payé, article, 9...","[on me demande 6 , 50 €de frais de retour !, j..."
2,2,581,2_pieds_bottes_pointure_pied,"[pieds, bottes, pointure, pied, confortables, ...",[de nombreux échange avec showroom et toujours...
3,3,543,3_mail_mails_réponse_aucune réponse,"[mail, mails, réponse, aucune réponse, envoyé,...","[mes mails sont restés sans réponse ., pour mo..."
4,4,457,4_vente privée_privée_vente_privées,"[vente privée, privée, vente, privées, ventes ...","[vente privée n'est plus ce que c'était, pas c..."
...,...,...,...,...,...
515,515,237,515_vais_dit_fois_rien,"[vais, dit, fois, rien, appel, rappeler, atten...",NaN
516,516,194,516_taille_cm_tailles_trop,"[taille, cm, tailles, trop, petit, 38, trop pe...",NaN
517,517,101,517_site_commander site_site éviter_plus site,"[site, commander site, site éviter, plus site,...",NaN
518,518,247,518_service client_client_service_clients,"[service client, client, service, clients, ser...",NaN


In [59]:
# topics trouvés
models["multilingue_2"].get_topic_info().to_csv("./artifacts/bertopic/topics_top_words_phrases.csv", index=False, encoding="utf8")
models["multilingue_2"].get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,1,61,1_avant date_date_date prévue_avant date prévue,"[avant date, date, date prévue, avant date pré...","[arrivé avant la date prévue ., livraison avan..."
1,2,28,2_livraison plus_plus rapide_livraison plus ra...,"[livraison plus, plus rapide, livraison plus r...","[livraison plus rapide que prévue !, livraison..."
2,3,21,3_bien passé tout_passé tout_tout très bien_bi...,"[bien passé tout, passé tout, tout très bien, ...","[tout s'est très bien passé ., tout c'est très..."
3,4,21,4_recommande recommande_recommande_recommande ...,"[recommande recommande, recommande, recommande...","[je ne recommande pas ., je ne le recommande p..."
4,5,21,5_rapport qualité prix_rapport qualité_qualité...,"[rapport qualité prix, rapport qualité, qualit...","[bon rapport qualité prix ., bon rapport quali..."
...,...,...,...,...,...
196,197,175,197_nul_fuir_euros très_plus jamais,"[nul, fuir, euros très, plus jamais, moque mon...",NaN
197,198,145,198_rapide_livraison_très_qualité,"[rapide, livraison, très, qualité, livraison t...",NaN
198,199,32,199_peu_peu longue_livraison peu longue_livrai...,"[peu, peu longue, livraison peu longue, livrai...",NaN
199,200,109,200_livraison temps_délais_livraison_livraison...,"[livraison temps, délais, livraison, livraison...",NaN


In [57]:
# Mots-clés associés à un topic
models["français"].get_topic(1)

[('fuir', np.float64(0.009360780014971159)),
 ('voleurs', np.float64(0.006536055892552369)),
 ('mefiez site', np.float64(0.006482369029416102)),
 ('deux pieds', np.float64(0.006482369029416102)),
 ('déconseille', np.float64(0.006154332195534195)),
 ('euros très', np.float64(0.006110943383651499)),
 ('mefiez', np.float64(0.006110943383651499)),
 ('honte', np.float64(0.005802463867721284)),
 ('moque monde', np.float64(0.005643290670603633)),
 ('site fuir', np.float64(0.005350966900032679)),
 ('déconseille fortement', np.float64(0.005323751485776222)),
 ('bravo', np.float64(0.005034728530178279)),
 ('chemin', np.float64(0.005013290442626738)),
 ('site', np.float64(0.004958475523537399)),
 ('garantie', np.float64(0.0049334318566711605))]

In [58]:
# Mots-clés associés à un topic
models["multilingue"].get_topic(1)

[('euros', np.float64(0.014610914133020642)),
 ('50', np.float64(0.007289140282516761)),
 ('bon achat', np.float64(0.005672219830212208)),
 ('frais', np.float64(0.005338886535221998)),
 ('payé', np.float64(0.005196259268486484)),
 ('article', np.float64(0.005114712229288268)),
 ('99', np.float64(0.005028546320017942)),
 ('retour', np.float64(0.004866339700676376)),
 ('achat', np.float64(0.004624742993202379)),
 ('euro', np.float64(0.004410232866890986)),
 ('frais retour', np.float64(0.004264238898192449)),
 ('remboursement', np.float64(0.0037714733604076764)),
 ('payer', np.float64(0.003715378032467817)),
 ('bon', np.float64(0.0036115305747170686)),
 ('90', np.float64(0.0036032736458079494))]

In [253]:
fig1 = models["français"].visualize_topics(top_n_topics=10)
fig2 = models["multilingue"].visualize_topics(top_n_topics=10)

combined_fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Modèle 1 : Français", "Modèle 2 : Multilingue")
)

for trace in fig1['data']:
    combined_fig.add_trace(trace, row=1, col=1)

for trace in fig2['data']:
    combined_fig.add_trace(trace, row=1, col=2)

combined_fig.update_layout(
    title_text="Répartition des topics",
    showlegend=False,
    height=600,
    width=1000
)

combined_fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
models["français"].visualize_barchart(top_n_topics=10)

In [ ]:
models["multilingue"].visualize_barchart(top_n_topics=10)

In [ ]:
models["multilingue"].visualize_hierarchy()

In [ ]:
models["multilingue"].visualize_hierarchy()